# 🎓 Plataforma Colaborativa de Estudos

Este notebook está dividido em células pequenas — rode **uma de cada vez**, na ordem, com `Shift+Enter`. Se alguma der erro, você sabe exatamente qual parte falhou.

**Antes de começar:** crie um arquivo `.env` na raiz do repositório com:
```
DB_HOST=localhost
DB_PORT=5432
DB_NAME=exercicios_colaborativos
DB_USER=postgres
DB_PASSWORD=sua_senha
```

**Ordem de execução:**
1. Instalar bibliotecas
2. Imports
3. Conexão com o banco
4. Funções auxiliares
5. Tela de Usuários


## 2. Imports e configuração inicial

In [1]:
!pip install pandas sqlalchemy psycopg2-binary panel python-dotenv jupyter_bokeh

In [2]:
import os
from dotenv import load_dotenv

import pandas as pd
import psycopg2 as pg
import sqlalchemy
from sqlalchemy import create_engine
import panel as pn
import matplotlib.pyplot as plt

In [3]:
# Carrega as variaveis do arquivo .env
load_dotenv()

True

In [4]:
# Le as variaveis de ambiente
DB_HOST = os.getenv('DB_HOST')
DB_NAME = os.getenv('DB_NAME')
DB_USER = os.getenv('DB_USER')
DB_PASS = os.getenv('DB_PASSWORD')

In [5]:
# Cria conexao com psycopg2 usando as variaveis carregadas
con = pg.connect(host=DB_HOST, dbname=DB_NAME, user=DB_USER, password=DB_PASS)

In [6]:
# Define a string de conexao para o SQLAlchemy, utilizando as variaveis do .env
# Cria o objeto engine do SQLAlchemy que sera usado para conectar e executar comandos no banco

cnx = f'postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}/{DB_NAME}'

sqlalchemy.create_engine(cnx)

engine = create_engine(cnx)

In [7]:
# Executa a consulta SQL para buscar todos os
# registros da tabela 'Usuario' no banco PostgreSQL
# e carrega o resultado em um DataFrame do pandas

query = "select * from Usuario;"
df = pd.read_sql_query(query, cnx)

df

,id_usuario,email,p_nome,sobrenome,senha,modo_focado,foto,biografia
0,1,grazyele@alu.ufc.br,Maria,Grazyele,senha123,False,None,Estudante de SI apaixonada por banco de dados.
1,2,ivna@alu.ufc.br,Ivna,Leite,senha123,False,None,Gosto de resolver problemas complexos.
2,3,joaopedro@alu.ufc.br,João,Pedro,senha123,True,None,Curioso por tecnologia e inovação.
3,4,carlos@email.com,Carlos,Mendes,senha456,False,None,Professor de Matemática há 10 anos.
4,5,fernanda@email.com,Fernanda,Souza,senha456,False,None,Especialista em Física Quântica.
5,6,lucas@email.com,Lucas,Almeida,senha789,True,None,Estudante de Engenharia.
6,7,amanda@email.com,Amanda,Costa,senha789,False,None,Amo Cálculo e Álgebra Linear.
7,8,rafael@email.com,Rafael,Nunes,senha321,False,None,Colaborador voluntário da plataforma.
8,9,beatriz@email.com,Beatriz,Lima,senha321,True,None,Estudante de Computação.
9,10,thiago@email.com,Thiago,Ferreira,senha654,False,None,Colaborador especialista em Química.


In [8]:
# Inicializa as extensoes do Panel necessarias para exibir tabelas
# interativas (Tabulator) e notificacoes na interface grafica

pn.extension()
pn.extension('tabulator')
pn.extension(notifications=True)
pn.extension('matplotlib')

In [9]:
# campos de texto

# declare esta variavel para usar na consulta de campos em branco
flag = ''

# Cria widgets interativos para o usuario inserir ou selecionar dados:

nome = pn.widgets.TextInput(
    name="Nome",
    value='',
    placeholder='Digite o nome',
    disabled=False
)

sobrenome = pn.widgets.TextInput(
    name="Sobrenome",
    value='',
    placeholder='Digite o sobrenome',
    disabled=False
)

email = pn.widgets.TextInput(
    name="E-mail",
    value='',
    placeholder='Digite o e-mail',
    disabled=False
)

senha = pn.widgets.PasswordInput(
    name="Senha",
    value='',
    placeholder='Digite a senha',
    disabled=False
)

foto = pn.widgets.TextInput(
    name="Foto de Perfil (URL)",
    value='',
    placeholder='Opcional - cole a URL da imagem',
    disabled=False
)

biografia = pn.widgets.TextAreaInput(
    name="Biografia",
    value='',
    placeholder='Opcional',
    disabled=False
)

modo_focado = pn.widgets.Checkbox(name='Modo Focado', value=False)

tipo_perfil = pn.widgets.RadioBoxGroup(
    name='Tipo de Perfil', options=['Estudante', 'Colaborador', 'Estudante + Colaborador'])

curso = pn.widgets.TextInput(
    name="Curso (se Estudante)",
    value='',
    placeholder='Opcional',
    disabled=False
)

id_usuario_alvo = pn.widgets.IntInput(
    name="ID do Usuario (para Atualizar/Excluir)",
    value=0
)

In [10]:
# Cria quatro botoes para as acoes principais da aplicacao CRUD:
# Consultar, Inserir, Excluir e Atualizar registros no banco de dados

buttonConsultar = pn.widgets.Button(name='Consultar', button_type='default')

buttonInserir = pn.widgets.Button(name='Inserir', button_type='default')

buttonExcluir = pn.widgets.Button(name='Excluir', button_type='default')

buttonAtualizar = pn.widgets.Button(name='Atualizar', button_type='default')

In [11]:
def queryAll():
    """
    Consulta todos os registros da tabela 'Usuario' no banco de dados e retorna
    um widget Tabulator para exibicao interativa dos dados.

    Returns:
        pn.widgets.Tabulator: Widget que exibe a tabela com todos os dados da tabela 'Usuario'.
    """
    query = "select id_usuario, p_nome, sobrenome, email, modo_focado from Usuario"
    df = pd.read_sql_query(query, cnx)
    return pn.widgets.Tabulator(df)


def limpar_campos():
    """
    Limpa todos os campos do formulario, deixando-os prontos para uma nova operacao.
    """
    nome.value = ''
    sobrenome.value = ''
    email.value = ''
    senha.value = ''
    foto.value = ''
    biografia.value = ''
    curso.value = ''
    modo_focado.value = False
    tipo_perfil.value = None
    id_usuario_alvo.value = 0


# consultar
# neste exemplo o metodo de consulta usa o dataframe do pandas como retorno. Note que a flag e usada para ignorar quando um
# campo for null (condicao e sempre verdadeira).
def on_consultar():
    """
    Consulta registros na tabela 'Usuario' filtrando dinamicamente por ID, 
    E-mail ou Nome/Sobrenome dependendo de quais campos foram preenchidos.
    """
    try:
        # Base da consulta
        base_query = "SELECT id_usuario, p_nome, sobrenome, email, modo_focado FROM Usuario"
        filtros = []
        parametros = []

        # 1. Se o ID foi preenchido (maior que 0), a busca por ID tem prioridade máxima
        if id_usuario_alvo.value > 0:
            filtros.append("id_usuario = %s")
            parametros.append(id_usuario_alvo.value)
            
        # 2. Se não tem ID, verifica se o e-mail foi preenchido
        elif email.value_input.strip() != '':
            filtros.append("email = %s")
            parametros.append(email.value_input.strip())
            
        # 3. Se não tem ID nem E-mail, verifica se Nome ou Sobrenome foram digitados
        else:
            if nome.value_input.strip() != '':
                filtros.append("p_nome ILIKE %s")
                parametros.append(f"%{nome.value_input.strip()}%")
            if sobrenome.value_input.strip() != '':
                filtros.append("sobrenome ILIKE %s")
                parametros.append(f"%{sobrenome.value_input.strip()}%")

        # Se houver algum filtro, junta eles com 'AND', senão traz a tabela inteira
        if filtros:
            query = f"{base_query} WHERE {' AND '.join(filtros)}"
            # Usando o 'con' do psycopg2 para evitar problemas de formatação de string do SQL plano
            df = pd.read_sql_query(query, con, params=parametros)
        else:
            # Se tudo estiver em branco, traz todos
            df = pd.read_sql_query(base_query, con)

        return pn.widgets.Tabulator(df)
        
    except Exception as e:
        return pn.pane.Alert(f'Não foi possível consultar: {str(e)}')


def on_inserir():
    """
    Insere um novo registro na tabela 'Usuario' usando os valores dos widgets
    nome, sobrenome, email, senha, foto, biografia e modo_focado. Conforme o
    tipo de perfil escolhido, tambem insere em Estudante e/ou Colaborador.
    Ao final, limpa o formulario automaticamente.

    Returns:
        pn.widgets.Tabulator ou pn.pane.Alert: Tabela atualizada ou alerta em caso de erro.
    """
    try:
        cursor = con.cursor()
        cursor.execute("insert into Usuario(p_nome, sobrenome, email, senha, modo_focado, biografia, foto) VALUES (%s, %s, %s, %s, %s, %s, %s) RETURNING id_usuario",
                    (nome.value_input, sobrenome.value_input, email.value_input, senha.value, modo_focado.value, biografia.value_input or None, foto.value_input or None))
        novo_id = cursor.fetchone()[0]

        if tipo_perfil.value in ('Estudante', 'Estudante + Colaborador'):
            cursor.execute("insert into Estudante(id_usuario, curso) VALUES (%s, %s)",
                        (novo_id, curso.value_input or None))
        if tipo_perfil.value in ('Colaborador', 'Estudante + Colaborador'):
            cursor.execute("insert into Colaborador(id_usuario) VALUES (%s)", (novo_id,))

        cursor.query
        con.commit()
        pn.state.notifications.success(f'Usuario #{novo_id} inserido com sucesso!')
        limpar_campos()
        return queryAll()
    except Exception as e:
        cursor.execute("ROLLBACK")
        cursor.close()
        return pn.pane.Alert(f'Não foi possível inserir: {str(e)}')


def on_atualizar():
    """
    Atualiza os campos nome, sobrenome, modo_focado, biografia e foto do
    registro identificado pelo ID informado em id_usuario_alvo.
    Ao final, limpa o formulario automaticamente.

    Returns:
        pn.widgets.Tabulator ou pn.pane.Alert: Tabela atualizada ou alerta em caso de erro.
    """
    try:
        cursor = con.cursor()
        cursor.execute("UPDATE Usuario SET p_nome = %s, sobrenome = %s, email = %s, modo_focado = %s, biografia = %s, foto = %s WHERE id_usuario = %s",
           (nome.value_input, sobrenome.value_input, email.value_input, modo_focado.value, biografia.value_input or None, foto.value_input or None, id_usuario_alvo.value))
        cursor.query
        con.commit()
        pn.state.notifications.success('Usuario atualizado com sucesso!')
        limpar_campos()
        return queryAll()
    except Exception as e:
        cursor.execute("ROLLBACK")
        cursor.close()
        return pn.pane.Alert(f'Não foi possível atualizar: {str(e)}')


def on_excluir():
    """
    Exclui o registro da tabela 'Usuario' com o ID informado em id_usuario_alvo.
    Ao final, limpa o formulario automaticamente.

    Returns:
        pn.widgets.Tabulator ou pn.pane.Alert: Tabela atualizada ou alerta em caso de erro.
    """
    try:
        cursor = con.cursor()
        cursor.execute("delete from Usuario where id_usuario=%s", (id_usuario_alvo.value,))
        rows_deleted = cursor.rowcount
        con.commit()
        pn.state.notifications.success('Usuario excluido com sucesso!')
        limpar_campos()
        return queryAll()
    except:
        cursor.execute("ROLLBACK")
        cursor.close()
        return pn.pane.Alert('Não foi possível excluir!')

In [12]:
# Funcao que chama a acao correta (consultar, inserir, atualizar ou excluir)
# dependendo do botao que foi clicado (representado pelos parametros booleanos)

def table_creator(cons, ins, atu, exc):
    if cons:
        return on_consultar()
    if ins:
        return on_inserir()
    if atu:
        return on_atualizar()
    if exc:
        return on_excluir()

In [13]:
# Cria uma ligacao interativa (bind) entre os botoes e a funcao que executa a acao correspondente,
# atualizando a tabela na interface sempre que algum botao for clicado.

interactive_table = pn.bind(table_creator, buttonConsultar, buttonInserir, buttonAtualizar, buttonExcluir)

In [14]:
def criar_graficos_pandas(event=None):
    query_tipo = """
        SELECT CASE WHEN e.id_usuario IS NOT NULL AND c.id_usuario IS NOT NULL THEN 'Est+Col'
                    WHEN e.id_usuario IS NOT NULL THEN 'Estudante'
                    WHEN c.id_usuario IS NOT NULL THEN 'Colaborador'
                    ELSE 'Sem perfil' END AS tipo,
               COUNT(*) as qtd
        FROM Usuario u
        LEFT JOIN Estudante e ON u.id_usuario = e.id_usuario
        LEFT JOIN Colaborador c ON u.id_usuario = c.id_usuario
        GROUP BY tipo
    """
    query_focado = """
        SELECT CASE WHEN modo_focado THEN 'Ativado' ELSE 'Desativado' END AS status,
               COUNT(*) as qtd
        FROM Usuario
        GROUP BY modo_focado
    """

    def_tipo = pd.read_sql(query_tipo, engine)

    def_focado = pd.read_sql(query_focado, engine)

    fig1, ax1 = plt.subplots(figsize=(6, 5))

    fig2, ax2 = plt.subplots(figsize=(6, 5))

    if not def_tipo.empty:
        def_tipo.set_index('tipo').plot(
            kind='pie',
            y='qtd',
            ax=ax1,
            autopct='%1.1f%%',
            legend=False,
            title='Usuarios por Tipo de Perfil',
            ylabel=''
        )
    else:
        ax1.text(0.5, 0.5, "Sem dados", ha='center')

    if not def_focado.empty:
        def_focado.set_index('status').plot(
            kind='bar',
            y='qtd',
            ax=ax2,
            color='#4CAF50',
            rot=0,
            legend=False,
            title='Usuarios por Modo Focado'
        )
        ax2.set_xlabel('Modo Focado')
    else:
        ax2.text(0.5, 0.5, "Sem dados", ha='center')

    plt.close(fig1)
    plt.close(fig2)

    return pn.Column(
        pn.pane.Matplotlib(fig1, tight=True),
        pn.pane.Matplotlib(fig2, tight=True),
        name="Graficos"
    )

In [15]:
layout_crud = pn.Row(
    pn.Column(
        'Usuario CRUD',
        nome, sobrenome, email, senha, tipo_perfil, curso, foto, biografia, modo_focado, id_usuario_alvo,
        pn.Row(buttonConsultar, buttonInserir),
        pn.Row(buttonAtualizar, buttonExcluir),
    ),
    pn.Column(interactive_table),
    name="Gerenciamento"
)

layout_dashboard = pn.bind(criar_graficos_pandas, buttonInserir)

aba_principal = pn.Tabs(
    layout_crud,
    ("Dashboard (Pandas)", layout_dashboard)
)

aba_principal.servable()

BokehModel(combine_events=True, render_bundle={'docs_json': {'ce822ca7-cc9b-4d80-afd0-798dbe62f8b5': {'version…